# RISC-V processor comparison: benchmarks on PYNQ

Runs the project's benchmarks on one core's overlay and reads back the cycle counters. The steps are in
`fpga/README.md`; the register map is in `fpga/rtl/fpga_shell.sv`.

The FPGA shell copies the RTL-simulation testbench: the same 256 KiB RAM, 1-cycle latency, MMIO and
counters. So each run's **cycles and marked cycles should equal the RTL simulation exactly**, and the
notebook checks that against `sw/manifest.json`. A mismatch is a finding: report it, don't tune it away.

Set `CORE` (and optionally `CLK_MHZ`), then run all cells. Repeat for each core.

In [ ]:
CORE = "hazard3"      # hazard3 | ibex | veer_el2
CLK_MHZ = None        # None = the clock built into the overlay (fpga/vivado/build.tcl clk_mhz, default 50)

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())
import runner
board = runner.Board(CORE, clk_mhz=CLK_MHZ)
print("core", CORE, "| FCLK0", board.clk_mhz, "MHz | ID 0x%08x" % board.rd("ID"))

## Smoke test: hello

In [ ]:
r = board.run("hello")
print(r["console"])
print({k: r[k] for k in ("exit", "cycles", "marked_cycles", "matches_rtl_sim")})

## All benchmarks (hello, Dhrystone, CoreMark, riscv-tests subset)

In [ ]:
results = board.run_all()
fails = [r["bench"] for r in results if r["exit"] != "0"]
mism  = [r["bench"] for r in results if not r.get("matches_rtl_sim")]
print(len(results), "programs |", len(results) - len(fails), "passed (exit 0) |",
      len(results) - len(mism), "identical to RTL simulation")
print("failed:", fails or "none")
print("differs from RTL sim:", mism or "none")

In [ ]:
# Benchmarks. CoreMark/MHz and DMIPS/MHz depend on cycles only, so they don't depend on the clock.
# The paper's numbers come from scripts/collect.py after the JSON below is committed.
by = {r["bench"]: r for r in results}
cm, dh = by["coremark"], by["dhrystone"]
print("CoreMark : exit", cm["exit"], "| marked cycles", cm["marked_cycles"], "| RTL sim", cm["rtl_sim"]["marked_cycles"])
print("Dhrystone: exit", dh["exit"], "| marked cycles", dh["marked_cycles"], "| RTL sim", dh["rtl_sim"]["marked_cycles"])
print()
print(cm["console"])

## Save

This writes `results/<core>_<clk>MHz.json`. Copy it into the repo at
`results/raw/fpga/<core>/board_<clk>MHz/` and commit it.

In [ ]:
path = board.save(results)
print("saved", path)